# Stage 0 — WRDS schema exploration (scratch)

Interactive companion to `scripts/00_verify_schema.py`. Use this to poke at tables the
script has not been taught about; anything worth keeping graduates into
`config.yaml:schema_targets` and then into `docs/wrds_schema_notes.md`, which is generated.

**Do not record findings here.** This notebook is disposable; the notes file is not.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from vrp.config import load_config
from vrp.wrds_conn import describe_table, get_connection, resolve_table, table_profile

cfg = load_config()
db = get_connection(cfg)
cfg.wrds_username

## What libraries can this account actually see?

In [ ]:
libs = db.list_libraries()
[l for l in sorted(libs) if any(k in l for k in ("crsp", "optionm", "wrdsapps", "comp"))]

In [ ]:
sorted(db.list_tables(library="optionm"))[:60]

## Resolve one target by hand

The same call the verification script makes. Add a candidate to `schema_fallbacks` in
`config.yaml` once you find one that works.

In [ ]:
target = "wrdsapps.opcrsphist"
fallbacks = cfg["schema_fallbacks"].get(target, [])
resolved = resolve_table(target, fallbacks, conn=db)
print(resolved)
describe_table(resolved, conn=db) if resolved else None

In [ ]:
# Which link scores are trustworthy? Stage 2a's filter should come from this, not folklore.
db.raw_sql(
    "SELECT score, COUNT(*) AS n, COUNT(DISTINCT permno) AS n_permno, "
    "COUNT(DISTINCT secid) AS n_secid "
    f"FROM {resolved} GROUP BY score ORDER BY score"
)

## Q5 — is there a continuous dividend yield for *equities*?

In [ ]:
[t for t in sorted(db.list_tables(library="optionm"))
 if any(k in t.lower() for k in ("dvd", "divid", "distr", "yield"))]

## Scratch